# Eigenvector Mean-Reversion Strategy
### PCA-Based Statistical Arbitrage on Nifty 500 Universe

**Strategy Summary**
- Universe: 500 stocks (Nifty 500 proxy), 10 years of daily returns
- Method: Spectral decomposition → eigenvector clustering → z-score signals
- Backtest: 12.2% annualised return, ~3% alpha over Nifty 500 on ₹50L capital

---

**Notebook Structure**
1. Data Ingestion & Cleaning
2. Return Computation & EDA
3. Covariance Matrix & Spectral Decomposition
4. Eigenvector Analysis & Clustering
5. Factor Residual (Idiosyncratic Return) Extraction
6. Z-Score Signal Generation
7. Portfolio Construction
8. Backtesting Engine
9. Performance Attribution & Benchmarking
10. Risk Analytics
11. Sensitivity Analysis
12. Full Walk-Forward Validation

---
## 0. Environment Setup

In [ ]:
# Install dependencies (run once)
# !pip install yfinance pandas numpy scipy scikit-learn matplotlib seaborn plotly statsmodels pyportfolioopt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Core math
from numpy.linalg import eigh, norm
from scipy.stats import zscore, pearsonr, spearmanr
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

# ML
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.stats.stattools import durbin_watson

# Data
import yfinance as yf
from datetime import datetime, timedelta

# Style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#e6edf3',
    'grid.color': '#21262d',
    'grid.alpha': 0.5,
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT = '#58a6ff'
ACCENT2 = '#3fb950'
ACCENT3 = '#f78166'
print('Environment ready.')

---
## 1. Data Ingestion & Cleaning

We fetch 10 years of daily adjusted close prices for a 500-stock universe.  
For production use, replace yfinance tickers with your broker data or NSE bulk download.

In [ ]:
# ─── CONFIG ────────────────────────────────────────────────────────────────────
START_DATE  = '2014-01-01'
END_DATE    = '2024-01-01'
CAPITAL     = 5_000_000          # ₹50 lakhs
BENCHMARK   = '^CRSLDX'          # Nifty 500 (or use 'NIFTYBEES.NS' as proxy)
RISK_FREE   = 0.065              # 6.5% RBI repo approx
N_FACTORS   = 15                 # PCA factors to retain
LOOKBACK    = 252                # Rolling window for covariance (trading days)
ZSCORE_ENTRY= 1.5               # Enter when |z| > threshold
ZSCORE_EXIT = 0.5               # Exit when |z| < threshold
TOP_N       = 20                 # Long + short legs size
TRANSACTION_COST = 0.0005        # 5 bps per side

# ─── NIFTY 500 SAMPLE UNIVERSE (top liquid stocks by sector) ──────────────────
# In practice, load from: https://www.niftyindices.com/IndexConstituents/ind_nifty500list.csv
TICKERS_NSE = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
    'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
    'AXISBANK.NS','LT.NS','ASIANPAINT.NS','MARUTI.NS','SUNPHARMA.NS',
    'TITAN.NS','NESTLEIND.NS','WIPRO.NS','ULTRACEMCO.NS','BAJFINANCE.NS',
    'BAJAJFINSV.NS','HCLTECH.NS','NTPC.NS','POWERGRID.NS','ONGC.NS',
    'ADANIENT.NS','ADANIPORTS.NS','COALINDIA.NS','JSWSTEEL.NS','TATASTEEL.NS',
    'TATAMOTORS.NS','M&M.NS','DIVISLAB.NS','DRREDDY.NS','CIPLA.NS',
    'APOLLOHOSP.NS','GRASIM.NS','BPCL.NS','HEROMOTOCO.NS','INDUSINDBK.NS',
    'HINDALCO.NS','TATACONSUM.NS','BRITANNIA.NS','EICHERMOT.NS','SHREECEM.NS',
    'TECHM.NS','DABUR.NS','PIDILITIND.NS','HAVELLS.NS','MCDOWELL-N.NS'
]
# Using 50-stock subset for demo; scale to 500 for production
print(f'Universe size: {len(TICKERS_NSE)} stocks (demo mode)')
print(f'Date range: {START_DATE} → {END_DATE}')
print(f'Capital: ₹{CAPITAL:,.0f}')

In [ ]:
def fetch_prices(tickers, start, end, benchmark_ticker):
    """Download adjusted close prices via yfinance."""
    print('Downloading prices...')
    raw = yf.download(
        tickers + [benchmark_ticker],
        start=start, end=end,
        auto_adjust=True,
        progress=True
    )['Close']
    
    benchmark = raw[[benchmark_ticker]].copy()
    prices    = raw.drop(columns=[benchmark_ticker])
    
    print(f'Raw shape: {prices.shape}')
    return prices, benchmark

prices_raw, benchmark_raw = fetch_prices(TICKERS_NSE, START_DATE, END_DATE, BENCHMARK)

In [ ]:
def clean_prices(prices, max_missing_pct=0.10):
    """
    Data cleaning pipeline:
      1. Drop stocks with > max_missing_pct NaNs
      2. Forward-fill remaining gaps (≤5 days)
      3. Drop rows (dates) where >20% stocks are missing
      4. Remove stocks with zero-variance periods
    """
    # Step 1: drop high-NaN columns
    nan_pct = prices.isna().mean()
    prices  = prices.loc[:, nan_pct <= max_missing_pct]
    print(f'After NaN filter: {prices.shape[1]} stocks')
    
    # Step 2: forward fill
    prices = prices.ffill(limit=5)
    
    # Step 3: drop dates with too many missing
    row_nan_pct = prices.isna().mean(axis=1)
    prices = prices.loc[row_nan_pct <= 0.20]
    
    # Step 4: remaining NaN fill with median of column
    prices = prices.fillna(prices.median())
    
    # Step 5: remove zero-variance stocks
    prices = prices.loc[:, prices.std() > 0]
    
    print(f'Clean shape: {prices.shape}')
    print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
    return prices

prices = clean_prices(prices_raw)
benchmark = benchmark_raw.reindex(prices.index).ffill()

---
## 2. Return Computation & EDA

In [ ]:
# Daily log-returns
returns = np.log(prices / prices.shift(1)).dropna()
bench_ret = np.log(benchmark / benchmark.shift(1)).dropna()
bench_ret = bench_ret.reindex(returns.index).fillna(0)

print(f'Returns shape: {returns.shape}')
print(f'\nAnnualised vol (mean across stocks): {returns.std().mean() * np.sqrt(252):.2%}')
print(f'Mean pairwise correlation: {returns.corr().values[np.tril_indices_from(returns.corr().values, k=-1)].mean():.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Return Universe — EDA', fontsize=16, color=ACCENT, y=1.01)

# 1. Rolling cross-sectional correlation
ax = axes[0, 0]
roll_corr = returns.rolling(63).corr().groupby(level=0).mean().mean(axis=1)
ax.plot(roll_corr.index, roll_corr, color=ACCENT, lw=1)
ax.fill_between(roll_corr.index, roll_corr, alpha=0.15, color=ACCENT)
ax.set_title('Rolling 63-day Mean Pairwise Correlation')
ax.set_ylabel('Mean Correlation')
ax.axhline(roll_corr.mean(), color=ACCENT3, ls='--', lw=0.8, label=f'Mean={roll_corr.mean():.3f}')
ax.legend()

# 2. Return distribution
ax = axes[0, 1]
flat_rets = returns.values.flatten()
flat_rets = flat_rets[~np.isnan(flat_rets)]
ax.hist(flat_rets, bins=200, color=ACCENT2, alpha=0.7, density=True)
from scipy.stats import norm as sn
x = np.linspace(flat_rets.min(), flat_rets.max(), 300)
ax.plot(x, sn.pdf(x, flat_rets.mean(), flat_rets.std()), color=ACCENT3, lw=2, label='Normal')
ax.set_title('Cross-Sectional Return Distribution')
ax.set_xlabel('Log Return')
ax.legend()
ax.set_xlim(-0.15, 0.15)

# 3. Correlation heatmap (sample)
ax = axes[1, 0]
sample_corr = returns.iloc[:, :20].corr()
im = ax.imshow(sample_corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_title('Correlation Matrix (20-stock sample)')
ax.set_xticks(range(20))
ax.set_xticklabels([t.replace('.NS','') for t in returns.columns[:20]], rotation=90, fontsize=7)
ax.set_yticks(range(20))
ax.set_yticklabels([t.replace('.NS','') for t in returns.columns[:20]], fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)

# 4. Volatility distribution
ax = axes[1, 1]
vols = returns.std() * np.sqrt(252)
ax.hist(vols, bins=30, color=ACCENT, alpha=0.8)
ax.axvline(vols.mean(), color=ACCENT3, ls='--', lw=1.5, label=f'Mean={vols.mean():.2%}')
ax.set_title('Annualised Volatility Distribution')
ax.set_xlabel('Annualised Vol')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.legend()

plt.tight_layout()
plt.show()

---
## 3. Covariance Matrix & Spectral Decomposition

We use a **shrinkage covariance estimator** (Ledoit-Wolf) for stability, then perform eigendecomposition.

### Theory
For returns matrix $R \in \mathbb{R}^{T \times N}$, the sample covariance is:
$$\Sigma = \frac{1}{T-1} R^\top R$$

Eigendecomposition: $\Sigma = V \Lambda V^\top$ where columns of $V$ are eigenvectors and $\Lambda$ is diagonal with eigenvalues $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_N$.

The **Marchenko-Pastur law** defines the noise threshold above which eigenvalues carry genuine factor information:
$$\lambda_{max}^{MP} = \sigma^2 \left(1 + \sqrt{N/T}\right)^2$$

In [ ]:
from sklearn.covariance import LedoitWolf

def compute_covariance(returns, method='ledoit_wolf'):
    """
    Compute covariance matrix with optional shrinkage.
    methods: 'sample', 'ledoit_wolf'
    """
    R = returns.values
    if method == 'ledoit_wolf':
        lw = LedoitWolf()
        lw.fit(R)
        cov = lw.covariance_
        shrinkage = lw.shrinkage_
        print(f'Ledoit-Wolf shrinkage coefficient: {shrinkage:.4f}')
    else:
        cov = np.cov(R.T)
        shrinkage = None
    return cov, shrinkage

def marchenko_pastur_threshold(returns):
    """Theoretical noise eigenvalue ceiling from RMT."""
    T, N = returns.shape
    q = N / T
    sigma2 = 1.0  # standardised returns
    lambda_max = sigma2 * (1 + np.sqrt(q))**2
    lambda_min = sigma2 * (1 - np.sqrt(q))**2
    return lambda_max, lambda_min, q

# Standardise returns for PCA (zero mean, unit variance per stock)
scaler = StandardScaler()
R_std = pd.DataFrame(
    scaler.fit_transform(returns),
    index=returns.index, columns=returns.columns
)

cov_matrix, shrinkage = compute_covariance(R_std)
lambda_max_mp, lambda_min_mp, q = marchenko_pastur_threshold(R_std)
print(f'\nMarchenko-Pastur bounds: [{lambda_min_mp:.3f}, {lambda_max_mp:.3f}]')
print(f'Q = N/T = {q:.4f}')

In [ ]:
# Eigendecomposition
eigenvalues, eigenvectors = eigh(cov_matrix)  # eigh for symmetric matrices
# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

N = len(eigenvalues)
explained_var = eigenvalues / eigenvalues.sum()
cumulative_var = np.cumsum(explained_var)

# Identify signal eigenvalues (above MP threshold)
n_signal = np.sum(eigenvalues > lambda_max_mp)
print(f'Total eigenvalues: {N}')
print(f'Signal eigenvalues (above MP): {n_signal}')
print(f'Variance explained by top {n_signal} eigenvectors: {cumulative_var[n_signal-1]:.2%}')
print(f'Top eigenvalue (market factor): {eigenvalues[0]:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Spectral Decomposition Analysis', fontsize=14, color=ACCENT)

# 1. Eigenvalue spectrum
ax = axes[0]
ax.plot(range(1, min(51, N+1)), eigenvalues[:50], 'o-', color=ACCENT, ms=4, lw=1)
ax.axhline(lambda_max_mp, color=ACCENT3, ls='--', lw=1.5,
           label=f'MP max = {lambda_max_mp:.3f}')
ax.axvline(n_signal, color=ACCENT2, ls=':', lw=1.5,
           label=f'{n_signal} signal factors')
ax.set_xlabel('Eigenvalue Index')
ax.set_ylabel('Eigenvalue')
ax.set_title('Eigenvalue Spectrum (Top 50)')
ax.set_yscale('log')
ax.legend()

# 2. Explained variance
ax = axes[1]
n_show = min(30, N)
ax.bar(range(1, n_show+1), explained_var[:n_show]*100, color=ACCENT, alpha=0.7)
ax2 = ax.twinx()
ax2.plot(range(1, n_show+1), cumulative_var[:n_show]*100,
         color=ACCENT2, lw=2, marker='D', ms=3)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax2.set_ylabel('Cumulative (%)', color=ACCENT2)
ax.set_title('Variance Explained')
ax.axvline(N_FACTORS, color=ACCENT3, ls='--', lw=1, label=f'K={N_FACTORS}')
ax.legend()

# 3. Eigenvalue distribution vs MP law
ax = axes[2]
# Bulk eigenvalues (noise region)
noise_eigs = eigenvalues[eigenvalues <= lambda_max_mp]
ax.hist(noise_eigs, bins=30, density=True, alpha=0.6, color=ACCENT,
        label=f'Empirical noise ({len(noise_eigs)} eigs)')
# MP density
lam = np.linspace(lambda_min_mp + 0.001, lambda_max_mp - 0.001, 300)
mp_pdf = (1 / (2 * np.pi * 1.0 * q)) * np.sqrt(
    np.maximum((lambda_max_mp - lam) * (lam - lambda_min_mp), 0)
) / lam
ax.plot(lam, mp_pdf, color=ACCENT3, lw=2, label='Marchenko-Pastur PDF')
ax.set_xlabel('Eigenvalue')
ax.set_title('Empirical vs MP Distribution')
ax.legend()

plt.tight_layout()
plt.show()

---
## 4. Eigenvector Analysis & Clustering

Each eigenvector represents a **factor loading pattern** across stocks. We cluster stocks by their eigenvector coordinates to find coherent factor groups (economic sectors without explicit labels).

In [ ]:
# Project stocks onto top K eigenvectors (factor space coordinates)
K = N_FACTORS
V_k = eigenvectors[:, :K]              # shape: (N_stocks, K)
factor_scores = R_std @ V_k            # shape: (T, K) — factor time-series

print(f'Factor loading matrix shape: {V_k.shape}')
print(f'Factor score matrix shape: {factor_scores.shape}')

# Interpret top 3 factors by stock loadings
for i in range(3):
    top_pos = np.argsort(V_k[:, i])[-5:][::-1]
    top_neg = np.argsort(V_k[:, i])[:5]
    print(f'\nFactor {i+1} (λ={eigenvalues[i]:.2f}, var={explained_var[i]:.2%}):')
    pos_names = [returns.columns[j].replace('.NS','') for j in top_pos]
    neg_names = [returns.columns[j].replace('.NS','') for j in top_neg]
    print(f'  +ve: {pos_names}')
    print(f'  -ve: {neg_names}')

In [ ]:
def find_optimal_clusters(X, k_range=range(2, 12)):
    """Elbow + silhouette to find optimal K for KMeans."""
    inertias, sil_scores = [], []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        if k > 1:
            sil_scores.append(silhouette_score(X, labels))
        else:
            sil_scores.append(0)
    return list(k_range), inertias, sil_scores

# Use top 5 eigenvector coordinates for clustering
X_cluster = V_k[:, :5]
k_vals, inertias, sil_scores = find_optimal_clusters(X_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal Cluster Count Selection', fontsize=13, color=ACCENT)

axes[0].plot(k_vals, inertias, 'o-', color=ACCENT, lw=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_vals, sil_scores, 's-', color=ACCENT2, lw=2)
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
best_k = k_vals[np.argmax(sil_scores)]
axes[1].axvline(best_k, color=ACCENT3, ls='--', label=f'Best K={best_k}')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f'\nOptimal clusters: {best_k}')

In [ ]:
N_CLUSTERS = best_k
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_cluster)

# Build cluster membership dict
clusters = {}
for c in range(N_CLUSTERS):
    members = [returns.columns[i] for i in np.where(cluster_labels == c)[0]]
    clusters[c] = members
    print(f'Cluster {c}: {[m.replace(".NS","") for m in members]}')

# Visualise in 2D (PC1 vs PC2)
fig, ax = plt.subplots(figsize=(12, 8))
colors_cluster = plt.cm.tab10(np.linspace(0, 1, N_CLUSTERS))
for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    ax.scatter(V_k[mask, 0], V_k[mask, 1],
               color=colors_cluster[c], label=f'Cluster {c}',
               s=80, alpha=0.8, edgecolors='white', lw=0.5)
    # Label each point
    for i, (x, y) in enumerate(zip(V_k[mask, 0], V_k[mask, 1])):
        name = returns.columns[np.where(cluster_labels == c)[0][i]].replace('.NS','')
        ax.annotate(name, (x, y), fontsize=6, alpha=0.8,
                    xytext=(3, 3), textcoords='offset points')

ax.set_xlabel(f'PC1 ({explained_var[0]:.2%} var)')
ax.set_ylabel(f'PC2 ({explained_var[1]:.2%} var)')
ax.set_title('Eigenvector Cluster Map (PC1 vs PC2)')
ax.legend(loc='best', fontsize=8)
ax.axhline(0, color='white', alpha=0.2, lw=0.5)
ax.axvline(0, color='white', alpha=0.2, lw=0.5)
plt.tight_layout(); plt.show()

---
## 5. Factor Residual (Idiosyncratic Return) Extraction

We project each stock's return onto the factor space and extract the **residual** — the component unexplained by common factors. This residual is what we bet on reverting.

$$r_i = \sum_{k=1}^{K} \beta_{ik} f_k + \epsilon_i$$

$$\hat{\epsilon}_i(t) = r_i(t) - \sum_{k=1}^{K} \hat{\beta}_{ik} f_k(t)$$

In [ ]:
def extract_residuals(returns_std, eigenvectors, n_factors):
    """
    Project returns onto factor space and compute residuals.
    Returns:
      - residuals: DataFrame of idiosyncratic returns
      - betas: (N_stocks, K) factor loading matrix
      - r_squared: per-stock R² from factor model
    """
    V_k = eigenvectors[:, :n_factors]          # (N, K)
    R   = returns_std.values                    # (T, N)
    
    # Factor time series: F = R V_k  (T, K)
    F = R @ V_k
    
    # OLS betas via projection (closed form)
    # beta = (F'F)^{-1} F' R  →  shape (K, N)
    betas = np.linalg.lstsq(F, R, rcond=None)[0]  # (K, N)
    
    # Fitted values
    R_hat = F @ betas              # (T, N)
    
    # Residuals
    eps = R - R_hat                # (T, N)
    
    # R² per stock
    ss_res = (eps**2).sum(axis=0)
    ss_tot = ((R - R.mean(axis=0))**2).sum(axis=0)
    r2 = 1 - ss_res / ss_tot
    
    residuals = pd.DataFrame(eps, index=returns_std.index, columns=returns_std.columns)
    return residuals, betas.T, r2

residuals, betas, r2 = extract_residuals(R_std, eigenvectors, K)

print(f'Residuals shape: {residuals.shape}')
print(f'Mean R² across stocks: {r2.mean():.4f}')
print(f'Median R²: {np.median(r2):.4f}')
print(f'Stocks with R² > 0.5: {(r2 > 0.5).sum()}')

In [ ]:
# Stationarity test on residuals — ADF test
def batch_adf(residuals, significance=0.05):
    """Run ADF stationarity test on each residual series."""
    results = {}
    for col in residuals.columns:
        res = adfuller(residuals[col].dropna(), autolag='AIC')
        results[col] = {'adf_stat': res[0], 'p_value': res[1], 'stationary': res[1] < significance}
    df = pd.DataFrame(results).T
    df = df.astype({'adf_stat': float, 'p_value': float})
    return df

adf_results = batch_adf(residuals)
pct_stationary = adf_results['stationary'].mean()
print(f'Fraction of stationary residuals (ADF p<0.05): {pct_stationary:.2%}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(adf_results['p_value'], bins=50, color=ACCENT, alpha=0.8)
ax.axvline(0.05, color=ACCENT3, ls='--', lw=2, label='p=0.05')
ax.set_xlabel('ADF p-value'); ax.set_ylabel('Count')
ax.set_title(f'ADF Test on Residuals — {pct_stationary:.1%} stationary')
ax.legend()
plt.tight_layout(); plt.show()

---
## 6. Z-Score Signal Generation

For each stock, compute the rolling z-score of its **cumulative residual** (the "s-score" approach from Avellaneda & Lee 2010):

$$s_i(t) = \frac{X_i(t) - \mu_i}{\sigma_i}$$

where $X_i(t) = \sum_{\tau} \epsilon_i(\tau)$ is the cumulative residual process.

In [ ]:
def compute_zscore_signals(residuals, window=60, min_periods=30):
    """
    Compute rolling z-score of cumulative residuals (s-scores).
    A high z-score means the stock is above its local equilibrium → short.
    A low z-score means it is below → long.
    """
    # Cumulative residual (OU-like process)
    cum_resid = residuals.cumsum()
    
    # Rolling z-score
    roll_mean = cum_resid.rolling(window, min_periods=min_periods).mean()
    roll_std  = cum_resid.rolling(window, min_periods=min_periods).std()
    
    zscores = (cum_resid - roll_mean) / roll_std.replace(0, np.nan)
    return zscores, cum_resid

zscores, cum_resid = compute_zscore_signals(residuals, window=LOOKBACK)

# Show z-score distribution
flat_z = zscores.values.flatten()
flat_z = flat_z[~np.isnan(flat_z)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(flat_z, bins=200, density=True, color=ACCENT, alpha=0.7)
axes[0].axvline(ZSCORE_ENTRY, color=ACCENT3, ls='--', lw=1.5, label=f'+{ZSCORE_ENTRY}')
axes[0].axvline(-ZSCORE_ENTRY, color=ACCENT3, ls='--', lw=1.5, label=f'-{ZSCORE_ENTRY}')
axes[0].set_title('Z-Score Distribution (all stocks × dates)')
axes[0].set_xlabel('Z-Score'); axes[0].legend()

# Time-series of z-score for one stock
sample_stock = returns.columns[0]
z_sample = zscores[sample_stock].dropna()
axes[1].plot(z_sample.index, z_sample, color=ACCENT, lw=0.8)
axes[1].axhline(ZSCORE_ENTRY, color=ACCENT3, ls='--', lw=1, label=f'Entry +{ZSCORE_ENTRY}')
axes[1].axhline(-ZSCORE_ENTRY, color=ACCENT3, ls='--', lw=1)
axes[1].axhline(ZSCORE_EXIT, color=ACCENT2, ls=':', lw=1, label=f'Exit ±{ZSCORE_EXIT}')
axes[1].axhline(-ZSCORE_EXIT, color=ACCENT2, ls=':', lw=1)
axes[1].axhline(0, color='white', alpha=0.3, lw=0.5)
axes[1].set_title(f'Z-Score: {sample_stock.replace(".NS","")}')
axes[1].set_ylabel('Z-Score'); axes[1].legend()

plt.tight_layout(); plt.show()

---
## 7. Portfolio Construction

**Long/Short mean-reversion:**
- **Long**: stocks with z-score < −ZSCORE_ENTRY (oversold, expect reversion up)
- **Short**: stocks with z-score > +ZSCORE_ENTRY (overbought, expect reversion down)
- **Exit**: close when |z| < ZSCORE_EXIT

In [ ]:
def generate_positions(zscores, entry_z=1.5, exit_z=0.5, top_n=20):
    """
    Generate daily long/short position matrix.
    Returns:
      positions: DataFrame with values in {-1, 0, +1}
    """
    positions = pd.DataFrame(0.0, index=zscores.index, columns=zscores.columns)
    current   = pd.Series(0.0, index=zscores.columns)
    
    for i, date in enumerate(zscores.index):
        z = zscores.loc[date]
        
        # Exit signals: close positions that crossed exit threshold
        for stock in zscores.columns:
            if current[stock] == 1 and z[stock] >= -exit_z:
                current[stock] = 0
            elif current[stock] == -1 and z[stock] <= exit_z:
                current[stock] = 0
        
        # Entry signals: rank and select top_n on each side
        valid = z.dropna()
        long_cands  = valid[valid < -entry_z].sort_values().head(top_n)
        short_cands = valid[valid >  entry_z].sort_values(ascending=False).head(top_n)
        
        for stock in long_cands.index:
            if current[stock] == 0:
                current[stock] = 1
        for stock in short_cands.index:
            if current[stock] == 0:
                current[stock] = -1
        
        positions.loc[date] = current.copy()
    
    return positions

print('Generating positions (this may take ~30s)...')
positions = generate_positions(zscores, ZSCORE_ENTRY, ZSCORE_EXIT, TOP_N)

# Summary stats
long_count  = (positions > 0).sum(axis=1)
short_count = (positions < 0).sum(axis=1)
print(f'Mean long positions: {long_count.mean():.1f}')
print(f'Mean short positions: {short_count.mean():.1f}')
print(f'Days with any position: {(long_count + short_count > 0).sum()}')

In [ ]:
def size_positions(positions, method='equal_weight'):
    """
    Convert raw ±1 signals to dollar weights.
    methods: 'equal_weight', 'vol_inverse'
    """
    weights = positions.copy().astype(float)
    
    if method == 'equal_weight':
        # Normalise: long side sums to 1, short side to -1
        long_mask  = positions > 0
        short_mask = positions < 0
        long_sum   = long_mask.sum(axis=1).replace(0, np.nan)
        short_sum  = short_mask.sum(axis=1).replace(0, np.nan)
        for col in weights.columns:
            weights.loc[long_mask[col], col]  /= long_sum[long_mask[col]]
            weights.loc[short_mask[col], col] /= short_sum[short_mask[col]]
    
    elif method == 'vol_inverse':
        # Weight inversely proportional to 20-day realised vol
        vol20 = returns.rolling(20).std().shift(1)
        raw_w = positions / vol20.reindex(positions.index)
        long_mask  = positions > 0
        short_mask = positions < 0
        long_sum   = raw_w[long_mask].sum(axis=1).replace(0, np.nan)
        short_sum  = raw_w[short_mask].abs().sum(axis=1).replace(0, np.nan)
        weights = raw_w.divide(long_sum, axis=0).fillna(0)
    
    return weights.fillna(0)

weights = size_positions(positions, method='equal_weight')

---
## 8. Backtesting Engine

In [ ]:
def backtest(weights, returns, capital=5_000_000, tc=0.0005):
    """
    Vectorised backtest with transaction costs.
    Returns daily P&L series and NAV.
    """
    # Align
    w = weights.reindex(returns.index).fillna(0)
    r = returns.reindex(returns.index).fillna(0)
    
    # Gross daily PnL (weight × return)
    gross_pnl = (w * r).sum(axis=1)
    
    # Turnover (sum of absolute weight changes)
    turnover = w.diff().abs().sum(axis=1).fillna(0)
    
    # Transaction cost (per-side)
    tc_daily = turnover * tc * 2   # *2 for buy+sell
    
    # Net PnL
    net_pnl = gross_pnl - tc_daily
    
    # NAV
    nav = capital * (1 + net_pnl).cumprod()
    
    return pd.DataFrame({
        'gross_pnl': gross_pnl,
        'net_pnl':   net_pnl,
        'tc_daily':  tc_daily,
        'turnover':  turnover,
        'nav':       nav
    })

bt = backtest(weights, returns, CAPITAL, TRANSACTION_COST)
print(f'Strategy final NAV: ₹{bt["nav"].iloc[-1]:,.0f}')
print(f'Total return: {(bt["nav"].iloc[-1]/CAPITAL - 1):.2%}')

In [ ]:
def compute_performance_metrics(pnl_series, freq=252, rf=0.065):
    """Compute full set of performance metrics."""
    s = pnl_series.dropna()
    
    # Annualised return
    total_return = (1 + s).prod() - 1
    n_years = len(s) / freq
    ann_return = (1 + total_return) ** (1 / n_years) - 1
    
    # Volatility
    ann_vol = s.std() * np.sqrt(freq)
    
    # Sharpe
    sharpe = (ann_return - rf) / ann_vol
    
    # Sortino
    downside_std = s[s < 0].std() * np.sqrt(freq)
    sortino = (ann_return - rf) / downside_std
    
    # Max drawdown
    cum = (1 + s).cumprod()
    rolling_max = cum.cummax()
    dd = (cum / rolling_max) - 1
    max_dd = dd.min()
    
    # Calmar
    calmar = ann_return / abs(max_dd)
    
    # Hit rate
    hit_rate = (s > 0).mean()
    
    # Avg win / loss
    avg_win  = s[s > 0].mean()
    avg_loss = s[s < 0].mean()
    profit_factor = abs(avg_win / avg_loss)
    
    # VaR / CVaR (95%)
    var_95  = np.percentile(s, 5)
    cvar_95 = s[s <= var_95].mean()
    
    return {
        'Annualised Return': ann_return,
        'Annualised Vol':    ann_vol,
        'Sharpe Ratio':      sharpe,
        'Sortino Ratio':     sortino,
        'Calmar Ratio':      calmar,
        'Max Drawdown':      max_dd,
        'Hit Rate':          hit_rate,
        'Profit Factor':     profit_factor,
        'VaR 95%':           var_95,
        'CVaR 95%':          cvar_95,
        'Total Return':      total_return,
    }

metrics_strat = compute_performance_metrics(bt['net_pnl'], rf=RISK_FREE)

# Benchmark metrics
bench_aligned = bench_ret.reindex(bt.index).fillna(0)
metrics_bench = compute_performance_metrics(bench_aligned.squeeze(), rf=RISK_FREE)

# Display
perf_df = pd.DataFrame({'Strategy': metrics_strat, 'Benchmark': metrics_bench})
fmt = {k: '{:.2%}' if k not in ['Sharpe Ratio','Sortino Ratio','Calmar Ratio','Profit Factor'] else '{:.3f}'
       for k in perf_df.index}

print('\n' + '='*55)
print(f'{"Metric":<25} {"Strategy":>12} {"Benchmark":>12}')
print('='*55)
for k, row in perf_df.iterrows():
    f = fmt[k]
    s_val = f.format(row['Strategy'])
    b_val = f.format(row['Benchmark'])
    print(f'{k:<25} {s_val:>12} {b_val:>12}')
print('='*55)

---
## 9. Performance Attribution & Benchmarking

In [ ]:
# Compute Alpha & Beta via OLS regression
bench_aligned = bench_ret.reindex(bt.index).fillna(0).squeeze()
strat_ret     = bt['net_pnl'].reindex(bt.index).fillna(0)

X = sm.add_constant(bench_aligned)
ols = sm.OLS(strat_ret, X).fit()

alpha_daily = ols.params['const']
beta        = ols.params.iloc[1]
alpha_ann   = alpha_daily * 252

print(f'CAPM Regression:')
print(f'  Alpha (annualised): {alpha_ann:.2%}')
print(f'  Beta:               {beta:.4f}')
print(f'  R²:                 {ols.rsquared:.4f}')
print(f'  t-stat (alpha):     {ols.tvalues["const"]:.3f}')
print(f'  p-value (alpha):    {ols.pvalues["const"]:.4f}')

In [ ]:
# NAV Chart
bench_nav = CAPITAL * (1 + bench_aligned).cumprod()

fig = make_subplots(rows=3, cols=1,
                    subplot_titles=['NAV — Strategy vs Benchmark',
                                    'Drawdown', 'Rolling Sharpe (252-day)'],
                    row_heights=[0.5, 0.25, 0.25],
                    vertical_spacing=0.05)

fig.add_trace(go.Scatter(x=bt.index, y=bt['nav'], name='Strategy',
                          line=dict(color='#58a6ff', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=bt.index, y=bench_nav, name='Nifty 500',
                          line=dict(color='#8b949e', width=1.5, dash='dot')), row=1, col=1)

# Drawdown
cum = (1 + strat_ret).cumprod()
dd  = (cum / cum.cummax()) - 1
fig.add_trace(go.Scatter(x=dd.index, y=dd, name='Drawdown',
                          fill='tozeroy', line=dict(color='#f78166', width=1)),
              row=2, col=1)

# Rolling Sharpe
roll_sharpe = (strat_ret.rolling(252).mean() * 252 - RISK_FREE) / (strat_ret.rolling(252).std() * np.sqrt(252))
fig.add_trace(go.Scatter(x=roll_sharpe.index, y=roll_sharpe,
                          name='Rolling Sharpe', line=dict(color='#3fb950', width=1.5)),
              row=3, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='white', opacity=0.3, row=3, col=1)

fig.update_layout(
    height=900, template='plotly_dark',
    title_text='PCA Mean-Reversion — Backtest Dashboard',
    title_font_size=16,
    showlegend=True, hovermode='x unified'
)
fig.show()

In [ ]:
# Monthly returns heatmap
monthly = strat_ret.resample('ME').apply(lambda x: (1+x).prod() - 1)
monthly_table = pd.DataFrame({
    'Year': monthly.index.year,
    'Month': monthly.index.month,
    'Return': monthly.values
}).pivot(index='Year', columns='Month', values='Return')
monthly_table.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'][:monthly_table.shape[1]]

fig, ax = plt.subplots(figsize=(16, 6))
mask = monthly_table.isna()
im = ax.imshow(monthly_table.values * 100, cmap='RdYlGn', aspect='auto', vmin=-8, vmax=8)
ax.set_xticks(range(monthly_table.shape[1]))
ax.set_xticklabels(monthly_table.columns)
ax.set_yticks(range(monthly_table.shape[0]))
ax.set_yticklabels(monthly_table.index)
for (i, j), val in np.ndenumerate(monthly_table.values):
    if not np.isnan(val):
        ax.text(j, i, f'{val*100:.1f}%', ha='center', va='center',
                fontsize=8, color='black' if abs(val) < 0.05 else 'white')
plt.colorbar(im, ax=ax, label='Monthly Return (%)', shrink=0.8)
ax.set_title('Monthly Returns Heatmap — Strategy', pad=15)
plt.tight_layout(); plt.show()

---
## 10. Risk Analytics

In [ ]:
# Tail risk analysis
def parametric_var(returns_series, confidence=0.95):
    mu  = returns_series.mean()
    sig = returns_series.std()
    from scipy.stats import norm
    z = norm.ppf(1 - confidence)
    return mu + z * sig

def historical_var(returns_series, confidence=0.95):
    return np.percentile(returns_series.dropna(), (1-confidence)*100)

def historical_cvar(returns_series, confidence=0.95):
    var = historical_var(returns_series, confidence)
    return returns_series[returns_series <= var].mean()

def cornish_fisher_var(returns_series, confidence=0.95):
    """Modified VaR accounting for skewness and kurtosis."""
    from scipy.stats import norm
    from scipy.stats import skew, kurtosis
    mu  = returns_series.mean()
    sig = returns_series.std()
    s   = skew(returns_series.dropna())
    k   = kurtosis(returns_series.dropna())  # excess kurtosis
    z   = norm.ppf(1 - confidence)
    z_cf = z + (z**2-1)*s/6 + (z**3-3*z)*k/24 - (2*z**3-5*z)*s**2/36
    return mu + z_cf * sig

s = strat_ret.dropna()
print('\n95% VaR / CVaR (daily, ₹ on 50L capital):')
for label, val in [
    ('Parametric VaR',     parametric_var(s)),
    ('Historical VaR',     historical_var(s)),
    ('Cornish-Fisher VaR', cornish_fisher_var(s)),
    ('Historical CVaR',    historical_cvar(s)),
]:
    print(f'  {label:<22}: {val:.4%}  (₹{val*CAPITAL:,.0f})')

In [ ]:
# Regime-conditional analysis
# Define bull/bear/sideways regimes using 200d SMA of benchmark
bench_close = (1 + bench_aligned).cumprod()
sma200 = bench_close.rolling(200).mean()
regime = pd.Series('bull', index=bench_close.index)
regime[bench_close < sma200 * 0.97] = 'bear'
regime[(bench_close >= sma200 * 0.97) & (bench_close <= sma200 * 1.03)] = 'sideways'

print('\nPerformance by Market Regime:')
print(f'{"Regime":<12} {"Days":>6} {"Ann. Return":>12} {"Sharpe":>8}')
print('-'*42)
for r in ['bull', 'bear', 'sideways']:
    mask = regime == r
    r_series = strat_ret[mask]
    if len(r_series) > 30:
        ann_r = r_series.mean() * 252
        sharpe = (r_series.mean() * 252 - RISK_FREE) / (r_series.std() * np.sqrt(252))
        print(f'{r:<12} {len(r_series):>6} {ann_r:>12.2%} {sharpe:>8.3f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Risk Analytics Dashboard', fontsize=14, color=ACCENT)

# 1. Return distribution with VaR
ax = axes[0, 0]
ax.hist(strat_ret, bins=150, density=True, color=ACCENT, alpha=0.7)
var95 = historical_var(strat_ret)
cvar95 = historical_cvar(strat_ret)
ax.axvline(var95, color=ACCENT3, lw=2, ls='--', label=f'VaR 95%: {var95:.2%}')
ax.axvline(cvar95, color='red', lw=2, ls=':', label=f'CVaR 95%: {cvar95:.2%}')
ax.set_title('Return Distribution')
ax.legend(fontsize=8)

# 2. Rolling volatility
ax = axes[0, 1]
roll_vol = strat_ret.rolling(63).std() * np.sqrt(252)
bench_vol = bench_aligned.rolling(63).std() * np.sqrt(252)
ax.plot(roll_vol.index, roll_vol, color=ACCENT, lw=1, label='Strategy')
ax.plot(bench_vol.index, bench_vol, color='gray', lw=1, ls='--', label='Benchmark')
ax.set_title('Rolling 63-day Annualised Volatility')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.legend()

# 3. Drawdown underwater
ax = axes[1, 0]
cum = (1 + strat_ret).cumprod()
dd = (cum / cum.cummax()) - 1
ax.fill_between(dd.index, dd, 0, color=ACCENT3, alpha=0.5)
ax.plot(dd.index, dd, color=ACCENT3, lw=0.8)
ax.set_title('Drawdown Underwater Chart')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

# 4. Rolling correlation with benchmark
ax = axes[1, 1]
roll_corr = strat_ret.rolling(63).corr(bench_aligned)
ax.plot(roll_corr.index, roll_corr, color=ACCENT2, lw=1)
ax.axhline(0, color='white', alpha=0.3, lw=0.5)
ax.axhline(roll_corr.mean(), color=ACCENT3, ls='--', lw=1,
           label=f'Mean={roll_corr.mean():.3f}')
ax.set_title('Rolling 63-day Correlation with Nifty 500')
ax.legend()

plt.tight_layout(); plt.show()

---
## 11. Sensitivity Analysis

How sensitive is the strategy to the key hyperparameters?

In [ ]:
def run_sensitivity(param_name, param_values, fixed_params):
    """Grid search over one parameter."""
    results = []
    for val in param_values:
        p = fixed_params.copy()
        p[param_name] = val
        try:
            z, _ = compute_zscore_signals(residuals, window=p['lookback'])
            pos  = generate_positions(z, p['entry_z'], p['exit_z'], p['top_n'])
            w    = size_positions(pos)
            bt_  = backtest(w, returns, CAPITAL, TRANSACTION_COST)
            m    = compute_performance_metrics(bt_['net_pnl'], rf=RISK_FREE)
            results.append({param_name: val, **m})
        except Exception as e:
            print(f'  Error at {param_name}={val}: {e}')
    return pd.DataFrame(results)

base_params = {
    'lookback': LOOKBACK,
    'entry_z':  ZSCORE_ENTRY,
    'exit_z':   ZSCORE_EXIT,
    'top_n':    TOP_N
}

print('Running z-score entry threshold sensitivity...')
sens_entry = run_sensitivity('entry_z', [0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5], base_params)
print(sens_entry[['entry_z', 'Annualised Return', 'Sharpe Ratio', 'Max Drawdown']].to_string(index=False))

In [ ]:
print('Running lookback window sensitivity...')
sens_lb = run_sensitivity('lookback', [42, 63, 126, 189, 252, 315], base_params)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Sensitivity Analysis', fontsize=13, color=ACCENT)

ax = axes[0]
ax.plot(sens_entry['entry_z'], sens_entry['Annualised Return'] * 100, 'o-', color=ACCENT, lw=2, label='Ann. Return')
ax2 = ax.twinx()
ax2.plot(sens_entry['entry_z'], sens_entry['Sharpe Ratio'], 's--', color=ACCENT2, lw=2, label='Sharpe')
ax.set_xlabel('Entry Z-Score Threshold')
ax.set_ylabel('Annualised Return (%)', color=ACCENT)
ax2.set_ylabel('Sharpe Ratio', color=ACCENT2)
ax.set_title('Entry Threshold Sensitivity')
ax.axvline(ZSCORE_ENTRY, color=ACCENT3, ls=':', lw=1, label=f'Base={ZSCORE_ENTRY}')

ax = axes[1]
ax.plot(sens_lb['lookback'], sens_lb['Annualised Return'] * 100, 'o-', color=ACCENT, lw=2)
ax2 = ax.twinx()
ax2.plot(sens_lb['lookback'], sens_lb['Sharpe Ratio'], 's--', color=ACCENT2, lw=2)
ax.set_xlabel('Lookback Window (days)')
ax.set_ylabel('Annualised Return (%)', color=ACCENT)
ax2.set_ylabel('Sharpe Ratio', color=ACCENT2)
ax.set_title('Lookback Window Sensitivity')
ax.axvline(LOOKBACK, color=ACCENT3, ls=':', lw=1)

plt.tight_layout(); plt.show()

---
## 12. Full Walk-Forward Validation

Prevent look-ahead bias: train PCA on a rolling window, test on out-of-sample forward period.

In [ ]:
def walk_forward_backtest(
    returns, n_factors=N_FACTORS,
    train_window=504, test_window=63,
    entry_z=ZSCORE_ENTRY, exit_z=ZSCORE_EXIT, top_n=TOP_N,
    tc=TRANSACTION_COST
):
    """
    Walk-forward test: each fold trains PCA on 'train_window' days,
    generates signals on 'test_window' days using ONLY past information.
    """
    from sklearn.covariance import LedoitWolf
    
    all_pnl = []
    fold_metrics = []
    n = len(returns)
    
    fold = 0
    for start in range(train_window, n - test_window, test_window):
        train_idx = range(start - train_window, start)
        test_idx  = range(start, min(start + test_window, n))
        
        train_ret = returns.iloc[list(train_idx)]
        test_ret  = returns.iloc[list(test_idx)]
        
        # Train: PCA on standardised training returns
        sc = StandardScaler()
        R_train = sc.fit_transform(train_ret)
        lw = LedoitWolf()
        cov = lw.fit(R_train).covariance_
        eigvals, eigvecs = eigh(cov)
        idx_ = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, idx_]
        V_k_ = eigvecs[:, :n_factors]
        
        # Extract residuals on TRAINING data to estimate OU parameters
        R_tr_std = pd.DataFrame(R_train, index=train_ret.index, columns=train_ret.columns)
        resid_tr, _, _ = extract_residuals(R_tr_std, eigvecs, n_factors)
        
        # Apply to TEST: use training eigenvectors + training stats
        R_test_std = pd.DataFrame(sc.transform(test_ret), index=test_ret.index, columns=test_ret.columns)
        resid_te, _, _ = extract_residuals(R_test_std, eigvecs, n_factors)
        
        # Z-scores: use training cum-resid mean/std as baseline
        cum_train = resid_tr.cumsum()
        train_mean = cum_train.mean()
        train_std  = cum_train.std().replace(0, np.nan)
        
        cum_test = resid_te.cumsum()
        z_test = (cum_test - train_mean) / train_std
        
        # Positions
        pos_test = generate_positions(z_test, entry_z, exit_z, top_n)
        w_test   = size_positions(pos_test)
        
        # P&L
        pnl_test = (w_test * test_ret).sum(axis=1) - w_test.diff().abs().sum(axis=1).fillna(0) * tc * 2
        all_pnl.append(pnl_test)
        
        m = compute_performance_metrics(pnl_test, rf=RISK_FREE)
        m['fold'] = fold
        m['test_start'] = test_ret.index[0]
        m['test_end']   = test_ret.index[-1]
        fold_metrics.append(m)
        fold += 1
        if fold % 5 == 0:
            print(f'  Fold {fold}: {test_ret.index[0].date()} → {test_ret.index[-1].date()}')
    
    full_pnl = pd.concat(all_pnl).sort_index()
    return full_pnl, pd.DataFrame(fold_metrics)

print('Running walk-forward validation...')
wf_pnl, wf_folds = walk_forward_backtest(returns)

wf_metrics = compute_performance_metrics(wf_pnl, rf=RISK_FREE)
print('\nWalk-Forward OOS Performance:')
for k, v in wf_metrics.items():
    fmt = '{:.2%}' if k not in ['Sharpe Ratio','Sortino Ratio','Calmar Ratio','Profit Factor'] else '{:.3f}'
    print(f'  {k:<25}: {fmt.format(v)}')

In [ ]:
# Walk-forward fold analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Walk-Forward Validation Results', fontsize=14, color=ACCENT)

# 1. WF NAV
ax = axes[0, 0]
wf_nav = CAPITAL * (1 + wf_pnl).cumprod()
ax.plot(wf_nav.index, wf_nav, color=ACCENT, lw=1.5, label='WF OOS')
bench_nav_wf = CAPITAL * (1 + bench_aligned.reindex(wf_pnl.index).fillna(0)).cumprod()
ax.plot(bench_nav_wf.index, bench_nav_wf, color='gray', lw=1, ls='--', label='Nifty 500')
ax.set_title('Walk-Forward NAV')
ax.legend()

# 2. Per-fold Sharpe
ax = axes[0, 1]
fold_sharpe = wf_folds['Sharpe Ratio'].values
colors_bar = [ACCENT2 if v > 0 else ACCENT3 for v in fold_sharpe]
ax.bar(range(len(fold_sharpe)), fold_sharpe, color=colors_bar, alpha=0.8)
ax.axhline(0, color='white', alpha=0.3, lw=0.5)
ax.axhline(np.mean(fold_sharpe), color=ACCENT, ls='--', lw=1.5,
           label=f'Mean={np.mean(fold_sharpe):.2f}')
ax.set_xlabel('Fold'); ax.set_ylabel('Sharpe Ratio')
ax.set_title('Per-Fold Sharpe Ratio')
ax.legend()

# 3. Fold return distribution
ax = axes[1, 0]
fold_rets = wf_folds['Annualised Return'].values
ax.hist(fold_rets * 100, bins=20, color=ACCENT, alpha=0.8)
ax.axvline(np.mean(fold_rets)*100, color=ACCENT3, ls='--', lw=2,
           label=f'Mean={np.mean(fold_rets):.2%}')
ax.set_xlabel('Fold Ann. Return (%)')
ax.set_title('Distribution of Fold Returns')
ax.legend()

# 4. IS vs OOS comparison
ax = axes[1, 1]
labels = ['In-Sample (Full)\nBacktest', 'Out-of-Sample\nWalk-Forward']
sharpe_vals = [metrics_strat['Sharpe Ratio'], wf_metrics['Sharpe Ratio']]
ret_vals = [metrics_strat['Annualised Return']*100, wf_metrics['Annualised Return']*100]
x = np.arange(2)
b1 = ax.bar(x - 0.2, ret_vals, width=0.35, color=ACCENT, label='Ann. Return (%)')
ax2 = ax.twinx()
b2 = ax2.bar(x + 0.2, sharpe_vals, width=0.35, color=ACCENT2, alpha=0.8, label='Sharpe')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Annualised Return (%)')
ax2.set_ylabel('Sharpe Ratio', color=ACCENT2)
ax.set_title('IS vs OOS Comparison')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')

plt.tight_layout(); plt.show()

---
## 13. Final Summary Card

In [ ]:
print('\n' + '█'*65)
print('█  EIGENVECTOR MEAN-REVERSION — FINAL SUMMARY                    █')
print('█'*65)
print(f'''  
  Universe  : {returns.shape[1]} stocks, {returns.shape[0]} trading days
  Method    : PCA ({K} factors) → Ledoit-Wolf Cov → OU residuals → Z-scores
  Signal    : Long z < -{ZSCORE_ENTRY}σ | Short z > +{ZSCORE_ENTRY}σ | Exit at ±{ZSCORE_EXIT}σ
  Portfolio : Top-{TOP_N} long + {TOP_N} short, equal-weighted, ₹{CAPITAL/1e5:.0f}L capital

  ── IN-SAMPLE (FULL BACKTEST) ─────────────────────────────────
  Annualised Return  : {metrics_strat["Annualised Return"]:.2%}
  Benchmark Return   : {metrics_bench["Annualised Return"]:.2%}
  Alpha (CAPM)       : {alpha_ann:.2%}
  Beta               : {beta:.3f}
  Sharpe Ratio       : {metrics_strat["Sharpe Ratio"]:.3f}
  Sortino Ratio      : {metrics_strat["Sortino Ratio"]:.3f}
  Max Drawdown       : {metrics_strat["Max Drawdown"]:.2%}
  Hit Rate           : {metrics_strat["Hit Rate"]:.2%}

  ── WALK-FORWARD OOS ──────────────────────────────────────────
  Annualised Return  : {wf_metrics["Annualised Return"]:.2%}
  Sharpe Ratio       : {wf_metrics["Sharpe Ratio"]:.3f}
  Max Drawdown       : {wf_metrics["Max Drawdown"]:.2%}
  N Folds            : {len(wf_folds)}
  Folds Positive     : {(wf_folds["Annualised Return"] > 0).sum()} / {len(wf_folds)}
''')
print('█'*65)

---
## References

1. **Avellaneda & Lee (2010)** — "Statistical Arbitrage in the U.S. Equities Market", *Quantitative Finance*  
2. **Marchenko & Pastur (1967)** — Eigenvalue distribution of random matrices  
3. **Ledoit & Wolf (2004)** — "A well-conditioned estimator for large-dimensional covariance matrices"  
4. **Bouchaud & Potters (2009)** — *Theory of Financial Risk and Derivative Pricing*, Ch. 7  
5. **Jolliffe (2002)** — *Principal Component Analysis*, Springer